# Phylogenetic Exploration — PEP725-temp (All species)

The `pep725_temp` data dump at `data/observations/pep725_temp/original/` contains the full PEP725 catalogue — **121 species** across 26 countries. This notebook mirrors the analysis in `phylogeny_fruit_trees.ipynb` but runs it across the whole catalogue, and then on a **deciduous-tree subset**.

Structure:
1. Build an `Observations` object directly from `pep725_temp` (bypassing the registered `PEP725Source`).
2. Select flowering (`BBCH_60`) observations and drop species that never flower in this data.
3. Run phylogenetic analysis — distance matrix, dendrogram, MDS, and Mantel test for a phenology signal.
4. Restrict to deciduous trees (hand-curated list) and rerun the same analysis.

> **Requires internet access** for the OpenTree API calls in §4 and §9.

In [ ]:
import re
import tarfile
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib import colormaps
from scipy.cluster import hierarchy as sch
from scipy.spatial.distance import squareform, pdist
from tqdm import tqdm

from pysephone.dataset.observations import Observations
from pysephone.dataset.util.phylogeny import PhylogenyFeatures
from pysephone.constants import (
    KEY_DATA_SOURCE, KEY_LOC_ID, KEY_YEAR,
    KEY_SPECIES_ID, KEY_SUBGROUP_ID, KEY_OBS_TYPE,
    KEY_OBSERVATIONS, KEY_LAT, KEY_LON,
)
from pysephone.paths import get_repo_root


def _palette(name: str, n: int):
    """Return a list of ``n`` RGB tuples drawn evenly from matplotlib colormap *name*."""
    cmap = colormaps.get_cmap(name)
    if hasattr(cmap, 'colors') and cmap.N >= n:
        return [cmap.colors[i % cmap.N] for i in range(n)]
    return [cmap(i / max(n - 1, 1)) for i in range(n)]


plt.rcParams.update({
    'figure.dpi': 120,
    'font.size': 10,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

## 1. Load `pep725_temp`

`pep725_temp/original/` is a flat directory of `.tar.gz` files named `PEP725_<CC>_<spc>_<sub>.tar.gz`. We:
1. parse those filenames to get the (country, species, subgroup) triples,
2. extract each archive on first use into `pep725_temp/extracted/`,
3. read the observations CSV (`PEP_ID;BBCH;YEAR;DAY`) and stations CSV,
4. look up scientific names from `PEP_options.txt` in the same folder.

The result is an `Observations` object using `src='pep725_temp'` — compatible with every downstream tool in pysephone.

This is scoped to this notebook (hence "temp"); nothing is written to the registry.

In [ ]:
SRC_KEY = 'pep725_temp'
TEMP_ROOT = get_repo_root() / 'data' / 'observations' / 'pep725_temp'
ORIG_DIR  = TEMP_ROOT / 'original'
EXTRACT_DIR = TEMP_ROOT / 'extracted'
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

# --- Name cleanup for OpenTree TNRS -------------------------------------
# PEP_options.txt occasionally encodes synonyms parenthetically
# ("Picea abies (P.excelsa)") or uses tautonyms that TNRS cannot resolve
# ("Rubus rubus"). We strip parentheticals and override known-bad names
# before the scientific name ever reaches PhylogenyFeatures.
_PAREN_RE = re.compile(r'\s*\([^)]*\)\s*')
_MANUAL_OVERRIDES = {
    'Rubus rubus': 'Rubus fruticosus',  # pepxx 207 = generic Rubus; use blackberry (most common in PEP725)
}

def clean_species_name(name: str) -> str:
    name = _PAREN_RE.sub(' ', name).strip()
    name = re.sub(r'\s+', ' ', name)
    return _MANUAL_OVERRIDES.get(name, name)

# --- Species names from PEP_options.txt ---------------------------------
_OPT_RE = re.compile(r'^\s*(\d+)\s*,\s*(.+?)\s*$')
species_name_by_code = {}
for line in (TEMP_ROOT / 'PEP_options.txt').read_text(encoding='utf-8').splitlines():
    m = _OPT_RE.match(line)
    if m:
        raw = m.group(2).replace('?', 'x').strip()
        species_name_by_code[int(m.group(1))] = clean_species_name(raw)

print(f'Species in PEP_options.txt: {len(species_name_by_code)}')

# --- Parse filenames -----------------------------------------------------
_FN_RE = re.compile(r'^PEP725_([A-Z]{2})_(\d{3})_(\d{3})\.tar\.gz$')
entries = []   # list of (country_code, species_code, subgroup_code, path)
for p in sorted(ORIG_DIR.iterdir()):
    m = _FN_RE.match(p.name)
    if m:
        entries.append((m.group(1), int(m.group(2)), int(m.group(3)), p))

print(f'Archive files:              {len(entries)}')
print(f'Distinct species codes:     {len(set(e[1] for e in entries))}')
print(f'Distinct country codes:     {len(set(e[0] for e in entries))}')

# Print any rewrites so the corrections are visible in the notebook output
print('\nName corrections applied:')
for code, clean in sorted(species_name_by_code.items()):
    raw = None
    for line in (TEMP_ROOT / 'PEP_options.txt').read_text(encoding='utf-8').splitlines():
        mm = _OPT_RE.match(line)
        if mm and int(mm.group(1)) == code:
            raw = mm.group(2).replace('?', 'x').strip()
            break
    if raw and raw != clean:
        print(f'  {code:>3d}  {raw!r:<36s}  →  {clean!r}')

In [ ]:
_WIN_BAD_CHARS = re.compile(r'[<>:"/\\|?*]')


def _extract(archive_path: Path, species_code: int, country_code: str) -> Path:
    """Extract one archive into extracted/<species>/<CC>_<species>_<sub>/ — idempotent.

    Some archives contain member names with characters that are illegal on
    Windows (``|``, ``?``, …). We rewrite the member name before extracting.

    A previous partial extraction is detected by missing marker files and
    replaced; the ``_ok`` sentinel is written on success so subsequent runs
    short-circuit safely.
    """
    target = EXTRACT_DIR / str(species_code) / archive_path.stem.replace('.tar', '')
    marker = target / '_ok'
    if marker.exists():
        return target
    # Partial extraction from an earlier failure — start clean
    if target.exists():
        import shutil
        shutil.rmtree(target)
    target.mkdir(parents=True, exist_ok=True)
    with tarfile.open(archive_path, 'r:gz') as tf:
        for member in tf.getmembers():
            safe_name = _WIN_BAD_CHARS.sub('_', member.name)
            if safe_name != member.name:
                member.name = safe_name
            tf.extract(member, path=target, filter='data')
    marker.write_text('ok')
    return target


def _load_stations(folder: Path, country_code: str) -> pd.DataFrame:
    """Read a stations CSV, fixing rows where NAME contains stray semicolons."""
    from io import StringIO
    path = folder / f'PEP725_{country_code}_stations.csv'
    raw = path.read_text(encoding='utf-8', errors='replace').splitlines()
    fixed = []
    for line in raw:
        parts = line.split(';')
        if len(parts) > 6:
            parts = parts[:5] + [' '.join(parts[5:])]
        fixed.append(';'.join(parts))
    df = pd.read_csv(StringIO('\n'.join(fixed)), sep=';', index_col='PEP_ID')
    df = df.rename(columns={'LAT': 'lat', 'LON': 'lon', 'NAME': 'loc_name'})
    df['country_code'] = country_code
    return df[['lat', 'lon', 'loc_name', 'country_code']]


def _load_observations(folder: Path, country_code: str,
                       species_code: int, subgroup_code: int):
    """Return a DataFrame of observations, or None if the folder has no CSV."""
    files = [p for p in folder.iterdir()
             if p.name.startswith(f'PEP725_{country_code}_')
             and p.name != f'PEP725_{country_code}_stations.csv']
    if len(files) != 1:
        return None
    df = pd.read_csv(files[0], sep=';')
    df = df.rename(columns={'PEP_ID': 'loc_id', 'YEAR': 'year', 'BBCH': 'obs_type'})
    df['obs_type'] = 'BBCH_' + df['obs_type'].astype(str)
    df['year']     = pd.to_numeric(df['year'], errors='coerce').astype('Int64')
    df['DAY']      = pd.to_numeric(df['DAY'],  errors='coerce')
    df = df.dropna(subset=['year', 'DAY'])
    df[KEY_OBSERVATIONS] = (
        pd.to_datetime(df['year'].astype(int).astype(str), errors='coerce')
        + pd.to_timedelta(df['DAY'].astype(int) - 1, unit='D')
    )
    df = df.dropna(subset=[KEY_OBSERVATIONS])
    if df.empty:
        return None
    df['species_id']  = species_code
    df['subgroup_id'] = subgroup_code
    df['year']        = df['year'].astype(int)
    return df[['loc_id', 'year', 'species_id', 'subgroup_id', 'obs_type', KEY_OBSERVATIONS]]


# ------------------------------------------------------------------------
# Extract everything we need and collect observations / stations
# ------------------------------------------------------------------------
obs_dfs = []
stations_by_country = {}

for cc, spc, sub, path in tqdm(entries, desc='Loading PEP725-temp'):
    try:
        folder = _extract(path, spc, cc)
    except Exception as exc:
        warnings.warn(f'extract failed for {path.name}: {exc}')
        continue
    if cc not in stations_by_country:
        try:
            stations_by_country[cc] = _load_stations(folder, cc)
        except Exception as exc:
            warnings.warn(f'station load failed for {cc}: {exc}')
    try:
        df_obs = _load_observations(folder, cc, spc, sub)
        if df_obs is not None:
            obs_dfs.append(df_obs)
    except Exception as exc:
        warnings.warn(f'obs load failed for {cc}/{spc}/{sub}: {exc}')

# ------------------------------------------------------------------------
# Build observations dataframe
# ------------------------------------------------------------------------
df_y = pd.concat(obs_dfs, ignore_index=True)
# Safety net: enforce datetime dtype in case of any upstream coercion
df_y[KEY_OBSERVATIONS] = pd.to_datetime(df_y[KEY_OBSERVATIONS], errors='coerce')
df_y = df_y.dropna(subset=[KEY_OBSERVATIONS])

df_y[KEY_DATA_SOURCE] = SRC_KEY
df_y = (
    df_y.set_index([KEY_DATA_SOURCE, 'loc_id', 'year',
                    'species_id', 'subgroup_id', 'obs_type'])
        [[KEY_OBSERVATIONS]]
)
df_y.index.names = [KEY_DATA_SOURCE, KEY_LOC_ID, KEY_YEAR,
                    KEY_SPECIES_ID, KEY_SUBGROUP_ID, KEY_OBS_TYPE]
# De-duplicate: the same (loc, year, species, subgroup, obs_type) can appear
# in multiple archives across countries because some stations live near borders.
df_y = df_y[~df_y.index.duplicated(keep='first')]

# ------------------------------------------------------------------------
# Build locations dataframe
# ------------------------------------------------------------------------
df_y_loc = pd.concat(stations_by_country.values())
df_y_loc = df_y_loc[~df_y_loc.index.duplicated(keep='first')].reset_index()
df_y_loc[KEY_DATA_SOURCE] = SRC_KEY
df_y_loc = df_y_loc.rename(columns={'PEP_ID': KEY_LOC_ID})
df_y_loc = df_y_loc.set_index([KEY_DATA_SOURCE, KEY_LOC_ID])
df_y_loc = df_y_loc[['lat', 'lon', 'loc_name', 'country_code']]
df_y_loc = df_y_loc[~df_y_loc.index.duplicated(keep='first')]

# ------------------------------------------------------------------------
# Build species_names dict  {(src, species_id): scientific_name}
# ------------------------------------------------------------------------
species_codes_present = sorted(set(df_y.index.get_level_values(KEY_SPECIES_ID)))
species_names = {
    (SRC_KEY, code): species_name_by_code.get(code, f'species_{code}')
    for code in species_codes_present
}

obs_all = Observations(df_y, df_y_loc, species_names=species_names)

print(f'\nTotal observations:  {len(obs_all):>8,}')
print(f'Unique species:       {len(species_codes_present):>8,}')
print(f'Unique stations:      {len(df_y_loc):>8,}')
print(f'Observation types:    {len(obs_all.observation_types)}')
print(f'Year range:           {df_y.index.get_level_values(KEY_YEAR).min()} - {df_y.index.get_level_values(KEY_YEAR).max()}')
print(f'Observations dtype:   {df_y[KEY_OBSERVATIONS].dtype}')

## 2. Flowering subset

Restrict to **BBCH_60** (beginning of flowering) — the same target used in `phylogeny_fruit_trees`. Species without BBCH_60 records in the data are dropped; they lack a comparable phenology target.

In [ ]:
TARGET_OBS = 'BBCH_60'

df_raw = obs_all._df_y
df_bloom = df_raw.xs(TARGET_OBS, level=KEY_OBS_TYPE).copy()
df_bloom['doy'] = df_bloom[KEY_OBSERVATIONS].dt.dayofyear
df_bloom = df_bloom.reset_index()
df_bloom['species_name'] = df_bloom[KEY_SPECIES_ID].map(
    {sid: name for (src, sid), name in species_names.items()}
)

counts = (df_bloom.groupby('species_name')['doy']
          .agg(n='count', mean='mean', std='std', median='median')
          .sort_values('median'))
print(f'Species with BBCH_60 records: {len(counts)}')
print()
print(counts.head(30).round(1).to_string())
print('...\n')
print(counts.tail(10).round(1).to_string())

In [ ]:
# Keep only species with enough flowering records to be statistically meaningful.
MIN_BLOOM_RECORDS = 30

species_with_bloom = counts[counts['n'] >= MIN_BLOOM_RECORDS].index.tolist()
df_bloom = df_bloom[df_bloom['species_name'].isin(species_with_bloom)].copy()

print(f'Species kept (n >= {MIN_BLOOM_RECORDS} BBCH_60 records): {len(species_with_bloom)}')
print(f'Bloom observations kept:                                 {len(df_bloom):,}')

# Reduce species_names to just the kept species so PhylogenyFeatures doesn't
# try to resolve species we have no phenology for.
species_names_bloom = {
    key: name for key, name in species_names.items()
    if name in species_with_bloom
}

# Site-year table for paired comparisons
df_site_year = (
    df_bloom
    .groupby([KEY_DATA_SOURCE, KEY_LOC_ID, KEY_YEAR, 'species_name'])['doy']
    .median()
    .reset_index()
)

# Rebuild a trimmed Observations (only kept species, only BBCH_60)
kept_sids = {k[1] for k in species_names_bloom}
mask = (
    df_raw.index.get_level_values(KEY_SPECIES_ID).isin(kept_sids)
    & (df_raw.index.get_level_values(KEY_OBS_TYPE) == TARGET_OBS)
)
df_y_kept = df_raw[mask]
obs_bloom = Observations(df_y_kept, df_y_loc, species_names=species_names_bloom)

## 3. Phenology distribution (all bloom species)

Before touching phylogeny: what does flowering look like across every species with enough BBCH_60 records?

In [ ]:
counts_kept = counts.loc[species_with_bloom].sort_values('median')
species_order = counts_kept.index.tolist()

_DOY_MONTH = [(1,'Jan'),(32,'Feb'),(60,'Mar'),(91,'Apr'),(121,'May'),(152,'Jun'),(182,'Jul')]

fig, ax = plt.subplots(figsize=(max(10, 0.30 * len(species_order)), 6))

doy_by_sp = [df_bloom.loc[df_bloom['species_name'] == sp, 'doy'].values
             for sp in species_order]
parts = ax.violinplot(doy_by_sp, positions=range(len(species_order)),
                      widths=0.85, showmedians=True, showextrema=False)
for pc in parts['bodies']:
    pc.set_facecolor('steelblue')
    pc.set_alpha(0.65)
parts['cmedians'].set_color('black')

ax.set_xticks(range(len(species_order)))
ax.set_xticklabels(species_order, rotation=75, ha='right', fontsize=7)
ax.set_ylabel('Day of year (BBCH_60 first bloom)')
ax.set_title(f'Flowering phenology across {len(species_order)} PEP725-temp species',
             fontweight='bold')
for doy, month in _DOY_MONTH:
    ax.axhline(doy, color='lightgrey', lw=0.5, zorder=0)
    ax.text(len(species_order) - 0.5, doy, month, fontsize=7, va='center', color='grey')
plt.tight_layout()
plt.show()

## 4. Reusable phylogenetic analysis

`run_phylo_analysis` encapsulates everything from `phylogeny_fruit_trees` into a single function so we can run it twice — once for the full PEP725-temp species set and once for the deciduous subset.

It returns a dict with the distance matrix, MDS coords, pair statistics, and the Mantel result.

In [ ]:
def _genus(name: str) -> str:
    return name.split()[0]


def _rankdata(x: np.ndarray) -> np.ndarray:
    tmp = np.argsort(x)
    ranks = np.empty_like(tmp, dtype=float)
    ranks[tmp] = np.arange(1, len(x) + 1)
    _, inv, counts_ = np.unique(x, return_inverse=True, return_counts=True)
    for i, c in enumerate(counts_):
        if c > 1:
            ranks[inv == i] = ranks[inv == i].mean()
    return ranks


def paired_pheno_diff(df_sy: pd.DataFrame, sp_i: str, sp_j: str):
    idx = [KEY_DATA_SOURCE, KEY_LOC_ID, KEY_YEAR]
    a = df_sy[df_sy['species_name'] == sp_i].set_index(idx)['doy']
    b = df_sy[df_sy['species_name'] == sp_j].set_index(idx)['doy']
    joined = a.to_frame('a').join(b.to_frame('b'), how='inner')
    if joined.empty:
        return np.nan, 0
    return (joined['a'] - joined['b']).abs().mean(), len(joined)


def mantel_test(D_phylo: np.ndarray, D_pheno: np.ndarray,
                n_perm: int = 9999, rng_seed: int = 0) -> dict:
    n = D_phylo.shape[0]
    idx = np.triu_indices(n, k=1)
    x, y = D_phylo[idx], D_pheno[idx]
    xr, yr = _rankdata(x), _rankdata(y)

    def _pearson(a, b):
        a, b = a - a.mean(), b - b.mean()
        denom = np.sqrt((a**2).sum() * (b**2).sum())
        return (a * b).sum() / denom if denom > 0 else 0.0

    r_obs, rho_obs = _pearson(x, y), _pearson(xr, yr)
    rng = np.random.default_rng(rng_seed)
    r_null = np.empty(n_perm)
    rho_null = np.empty(n_perm)
    for k in range(n_perm):
        perm = rng.permutation(n)
        yp = D_pheno[np.ix_(perm, perm)][idx]
        r_null[k]   = _pearson(x,  yp)
        rho_null[k] = _pearson(xr, _rankdata(yp))
    return {
        'pearson':  {'stat': r_obs,   'p_val': (r_null   >= r_obs).mean(),   'null': r_null},
        'spearman': {'stat': rho_obs, 'p_val': (rho_null >= rho_obs).mean(), 'null': rho_null},
    }


def _detect_orphan_species(D: np.ndarray, species_keys: list,
                           fallback_value: float = 1.0,
                           orphan_frac_threshold: float = 0.4) -> list:
    """Return species_keys whose rows in D look like PhylogenyFeatures fallbacks.

    Two patterns trigger an orphan classification:

    * **TNRS-unresolved** — every off-diagonal entry in the row equals ``D.max()``.
      These species are filled with the global max distance at the end of
      ``PhylogenyFeatures._compute_distances``.
    * **Not in induced subtree** — a fraction ``>= orphan_frac_threshold`` of
      off-diagonal entries equals ``fallback_value`` (the hardcoded ``d = 1.0``
      in ``PhylogenyFeatures._parse_and_compute``).
    """
    n = D.shape[0]
    if n < 3:
        return []
    max_d = D.max()
    orphans = []
    for i in range(n):
        row = np.delete(D[i], i)
        if np.allclose(row, max_d):
            orphans.append(species_keys[i])
            continue
        if np.isclose(row, fallback_value).mean() >= orphan_frac_threshold:
            orphans.append(species_keys[i])
    return orphans


def run_phylo_analysis(obs: Observations, df_sy: pd.DataFrame,
                       k_embed: int = 16, n_perm: int = 1999,
                       label: str = '', prune_orphans: bool = True) -> dict:
    """Fit PhylogenyFeatures, build paired-DOY matrix, run Mantel test.

    If *prune_orphans* is True, we fit once, detect species whose rows in the
    distance matrix are PhylogenyFeatures fallbacks (TNRS-unresolved or missing
    from the induced subtree), drop them from the observations, and refit. This
    removes the pile-ups at ``d = 1.0`` and ``d = max`` that otherwise swamp
    the Mantel correlation.
    """
    def _fit(obs_: Observations) -> PhylogenyFeatures:
        p = PhylogenyFeatures(k_embed=k_embed, output=['mds', 'distances'])
        with warnings.catch_warnings(record=True) as captured:
            warnings.simplefilter('always')
            p.fit(obs_)
        if captured:
            print(f'[{label}] warnings during fit:')
            for w in captured[:15]:
                print(f'  {w.message}')
            if len(captured) > 15:
                print(f'  … (+{len(captured) - 15} more)')
        return p

    phylo = _fit(obs)

    if prune_orphans:
        orphans = _detect_orphan_species(phylo.distance_matrix, phylo.species_keys)
        if orphans:
            print(f'[{label}] dropping {len(orphans)} orphan species '
                  f'(TNRS-unresolved or absent from induced subtree):')
            for key in orphans:
                print(f'  {key[1]:>4d}  {obs.species_names[key]}')
            kept_names = {k: v for k, v in obs.species_names.items() if k not in orphans}
            kept_sids = {k[1] for k in kept_names}
            df_full = obs._df_y
            mask = df_full.index.get_level_values(KEY_SPECIES_ID).isin(kept_sids)
            obs = Observations(df_full[mask], obs._df_y_loc, species_names=kept_names)
            phylo = _fit(obs)

    labels = {k: obs.species_names[k] for k in phylo.species_keys}
    label_list = [labels[k] for k in phylo.species_keys]
    D = phylo.distance_matrix
    n_sp = len(label_list)

    # Paired phenological differences
    pairs = []
    for i in range(n_sp):
        for j in range(i + 1, n_sp):
            sp_i, sp_j = label_list[i], label_list[j]
            diff, n = paired_pheno_diff(df_sy, sp_i, sp_j)
            if np.isnan(diff):
                continue
            pairs.append((sp_i, sp_j, i, j, D[i, j], diff, n))

    # Phenological distance matrix — Mantel requires no missing off-diagonal
    D_pheno = np.full((n_sp, n_sp), np.nan)
    for sp_i, sp_j, i, j, _, diff, _ in pairs:
        D_pheno[i, j] = D_pheno[j, i] = diff
    np.fill_diagonal(D_pheno, 0.0)
    if np.isnan(D_pheno).any():
        max_d = np.nanmax(D_pheno) if np.isfinite(np.nanmax(D_pheno)) else 1.0
        D_pheno = np.where(np.isnan(D_pheno), max_d, D_pheno)

    mantel = mantel_test(D, D_pheno, n_perm=n_perm)

    return {
        'label':      label,
        'phylo':      phylo,
        'labels':     labels,
        'label_list': label_list,
        'D':          D,
        'D_pheno':    D_pheno,
        'pairs':      pairs,
        'mantel':     mantel,
    }

## 5. Fit phylogeny for the full bloom species set

This fires a TNRS request + an induced-subtree request to OpenTree. With ~100 species the induced subtree can take a few seconds.

In [ ]:
result_all = run_phylo_analysis(obs_bloom, df_site_year, k_embed=16,
                                 n_perm=1999, label='all bloom species')

n_sp = len(result_all['label_list'])
print(f'\nSpecies in phylogeny: {n_sp}')
print(f'Pairs with shared site-years: {len(result_all["pairs"])} / {n_sp * (n_sp - 1) // 2}')
for key, res in result_all['mantel'].items():
    print(f"  Mantel {key:<9s}: stat={res['stat']:+.3f}  p={res['p_val']:.4f}")

## 6. Plots for the full set

Reusable plotting functions — distance heatmap, dendrogram, MDS scatter and quality, Mantel nulls, scatter of phylogenetic vs phenological distance.

In [ ]:
def plot_distance_matrix(result: dict, annotate: bool = None):
    D = result['D']
    lbl = result['label_list']
    n = len(lbl)
    if annotate is None:
        annotate = n <= 12
    fig, ax = plt.subplots(figsize=(min(14, 0.22 * n + 4), min(12, 0.22 * n + 3)))
    im = ax.imshow(D, cmap='viridis_r', aspect='auto')
    plt.colorbar(im, ax=ax, shrink=0.8, label='Patristic distance')
    ax.set_xticks(range(n))
    ax.set_yticks(range(n))
    ax.set_xticklabels(lbl, rotation=80, ha='right', fontsize=max(5, 11 - n // 10))
    ax.set_yticklabels(lbl,              fontsize=max(5, 11 - n // 10))
    if annotate:
        for i in range(n):
            for j in range(n):
                ax.text(j, i, f'{D[i,j]:.2f}', ha='center', va='center', fontsize=7,
                        color='white' if D[i,j] > D.max() * 0.5 else 'black')
    ax.set_title(f'Phylogenetic distance matrix — {result["label"]} ({n} species)',
                 fontweight='bold')
    plt.tight_layout()
    plt.show()


def plot_dendrogram(result: dict):
    D = result['D']
    lbl = result['label_list']
    n = len(lbl)
    linkage = sch.linkage(squareform(D, checks=False), method='average')
    genera = sorted(set(_genus(s) for s in lbl))
    palette = _palette('tab20', max(20, len(genera)))
    genus_colors = {g: palette[i % len(palette)] for i, g in enumerate(genera)}

    fig, ax = plt.subplots(figsize=(max(11, 0.20 * n + 4), max(5, min(10, 0.10 * n + 4))))
    sch.dendrogram(linkage, labels=lbl,
                   leaf_rotation=75, leaf_font_size=max(5, 11 - n // 10),
                   ax=ax, color_threshold=0, above_threshold_color='#555555')
    for t in ax.get_xmajorticklabels():
        t.set_color(genus_colors[_genus(t.get_text())])
        t.set_fontweight('semibold')
    ax.set_ylabel('Patristic distance')
    ax.set_title(f'UPGMA dendrogram — {result["label"]} ({n} species)', fontweight='bold')
    plt.tight_layout()
    plt.show()
    return linkage, genus_colors


def plot_mds(result: dict, linkage: np.ndarray, genus_colors: dict,
             max_dims: int = 4):
    coords = result['phylo'].mds_coords
    lbl = result['label_list']
    D = result['D']
    n = len(lbl)
    k_actual = min(coords.shape[1], max_dims)

    # Eigenspectrum
    D2 = D ** 2
    H  = np.eye(n) - np.ones((n, n)) / n
    B  = -0.5 * H @ D2 @ H
    B  = (B + B.T) / 2
    evals = np.linalg.eigvalsh(B)
    evals_sort = np.sort(np.maximum(evals, 0.0))[::-1]
    var_per_dim = evals_sort[:coords.shape[1]] / evals_sort.sum() * 100
    cumvar = np.cumsum(evals_sort) / evals_sort.sum() * 100
    frac_neg = (-np.minimum(evals, 0)).sum() / np.abs(evals).sum() * 100

    idx_upper = np.triu_indices(n, k=1)
    d_true = D[idx_upper]
    def _stress(ck):
        return np.sqrt(((d_true - pdist(ck)) ** 2).sum() / (d_true ** 2).sum())
    stress_vals = [_stress(coords[:, :k]) for k in range(1, coords.shape[1] + 1)]

    # --- MDS pairwise scatter ---
    fig, axes = plt.subplots(k_actual, k_actual,
                              figsize=(2.1 * k_actual, 2.1 * k_actual),
                              squeeze=False)
    fig.suptitle(f'MDS pairwise scatter — {result["label"]}', fontweight='bold')
    for row in range(k_actual):
        for col in range(k_actual):
            ax = axes[row][col]
            if row == col:
                ax.text(0.5, 0.5, f'MDS {row+1}\n{var_per_dim[row]:.1f}% var',
                        ha='center', va='center', fontsize=9, transform=ax.transAxes)
                ax.set_xticks([]); ax.set_yticks([])
            elif col > row:
                for name, pt in zip(lbl, coords):
                    ax.scatter(pt[col], pt[row], s=25 if n > 30 else 55,
                               color=genus_colors[_genus(name)],
                               edgecolors='white', linewidths=0.4, zorder=3)
                ax.set_xticks([]); ax.set_yticks([])
            else:
                ax.set_visible(False)
    plt.tight_layout()
    plt.show()

    # --- MDS diagnostics ---
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    fig.suptitle('MDS quality', fontweight='bold')
    ax = axes[0]
    kk = np.arange(1, coords.shape[1] + 1)
    ax.bar(kk, var_per_dim, color='steelblue', alpha=0.8)
    ax2 = ax.twinx()
    ax2.plot(kk, cumvar[:coords.shape[1]], 'o-', color='darkorange', lw=1.5)
    ax2.set_ylabel('Cumulative variance (%)', color='darkorange'); ax2.set_ylim(0, 105)
    ax.set_xlabel('MDS dim'); ax.set_ylabel('Variance (%)'); ax.set_title('Eigenspectrum')

    ax = axes[1]
    ax.plot(kk, stress_vals, 's-', color='crimson')
    ax.axhline(0.1, color='grey', ls='--', lw=1, label='Stress-1 = 0.1')
    ax.axhline(0.2, color='grey', ls=':',  lw=1, label='Stress-1 = 0.2')
    ax.set_xlabel('k dims'); ax.set_ylabel('Stress-1'); ax.set_title('Kruskal stress-1')
    ax.legend(fontsize=8)
    ax.text(0.97, 0.97, f'Non-Euclid mass: {frac_neg:.1f}%',
            ha='right', va='top', transform=ax.transAxes, fontsize=8, color='grey')

    ax = axes[2]
    ax.scatter(d_true, pdist(coords), s=15, alpha=0.5, color='steelblue', edgecolors='none')
    mn, mx = d_true.min(), d_true.max()
    ax.plot([mn, mx], [mn, mx], 'k--', lw=1)
    ax.set_xlabel('True patristic distance'); ax.set_ylabel('MDS distance')
    ax.set_title(f'Shepard  (Stress-1={stress_vals[-1]:.3f})')
    plt.tight_layout()
    plt.show()


def plot_phylo_signal(result: dict, label_pairs: bool = None):
    pairs = result['pairs']
    if not pairs:
        print(f'[{result["label"]}] no shared site-year pairs — skipping signal plot')
        return
    x = np.array([p[4] for p in pairs])
    y = np.array([p[5] for p in pairs])
    n = np.array([p[6] for p in pairs])
    if label_pairs is None:
        label_pairs = len(pairs) <= 40

    r = np.corrcoef(x, y)[0, 1]
    rho = np.corrcoef(_rankdata(x), _rankdata(y))[0, 1]

    fig, ax = plt.subplots(figsize=(8, 5.5))
    for sp_i, sp_j, _, _, pd_, phd, n_ in pairs:
        same_genus = _genus(sp_i) == _genus(sp_j)
        ax.scatter(pd_, phd,
                   color='#2ecc71' if same_genus else '#e74c3c',
                   s=20 + (n_ / max(n.max(), 1)) * 200,
                   alpha=0.7, edgecolors='white', linewidths=0.5, zorder=3)
        if label_pairs:
            ax.annotate(f'{sp_i.split()[0][0]}.{sp_i.split()[-1]}–\n'
                        f'{sp_j.split()[0][0]}.{sp_j.split()[-1]}',
                        (pd_, phd), xytext=(4, 2), textcoords='offset points',
                        fontsize=6, color='#444')

    m = np.polyfit(x, y, 1)
    xr = np.linspace(x.min(), x.max(), 100)
    ax.plot(xr, np.polyval(m, xr), 'k--', lw=1.3,
            label=f'Linear fit  (r={r:.2f}, ρ={rho:.2f})')

    ax.legend(handles=[
        mpatches.Patch(color='#2ecc71', label='Same genus'),
        mpatches.Patch(color='#e74c3c', label='Different genus'),
    ] + ax.get_legend_handles_labels()[0][-1:], fontsize=8.5)
    ax.set_xlabel('Phylogenetic distance (patristic)')
    ax.set_ylabel('Mean paired |ΔDOY| (days, same loc & year)')
    ax.set_title(f'Phylogenetic signal in flowering — {result["label"]}',
                 fontweight='bold')
    plt.tight_layout()
    plt.show()


def plot_mantel(result: dict):
    mantel = result['mantel']
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    fig.suptitle(f'Mantel null distributions — {result["label"]}', fontweight='bold')
    for ax, (key, res), color in zip(axes, mantel.items(),
                                      ['steelblue', 'darkorange']):
        ax.hist(res['null'], bins=50, color=color, alpha=0.65, edgecolor='none')
        ax.axvline(res['stat'], color='crimson', lw=2,
                   label=f"obs={res['stat']:+.3f}\np={res['p_val']:.4f}")
        ax.set_title(f'{"Pearson r" if key=="pearson" else "Spearman ρ"}  (p={res["p_val"]:.4f})')
        ax.set_xlabel(key); ax.set_ylabel('count')
        ax.legend(fontsize=9)
    plt.tight_layout()
    plt.show()

In [ ]:
plot_distance_matrix(result_all)

In [ ]:
linkage_all, genus_colors_all = plot_dendrogram(result_all)

In [ ]:
plot_mds(result_all, linkage_all, genus_colors_all, max_dims=4)

In [ ]:
plot_phylo_signal(result_all)
plot_mantel(result_all)

## 7. Deciduous tree subset

The PEP725-temp catalogue mixes woody trees and shrubs with herbaceous crops (wheat, barley, maize, …), evergreens (pine, spruce, olive, citrus, …), and a few evergreen shrubs. A "deciduous tree/shrub" signal is cleaner than the whole-catalogue one because:
- herbaceous species have fundamentally different phenology and growth strategies,
- evergreens don't undergo the leaf-out/dormancy cycle that ties bloom timing to winter chilling,
- within deciduous woody plants the trees share a common climate-driven flowering mechanism, so the phylogenetic signal should sharpen.

Below is a **hand-curated list** of deciduous tree / shrub genera present in the catalogue. Notable edge cases:
- `Larix decidua` is a deciduous conifer (included).
- `Prunus cerasifera`/`padus`/`cerasus` are deciduous Rosaceae (included).
- `Calluna`, `Arbutus`, `Cistus`, `Olea`, `Citrus`, `Laurus`, `Rosmarinus`, `Thymus`, `Ficus` are evergreen / semi-evergreen (excluded).
- `Hedera`, `Buxus` not present.
- `Vitis`, `Ribes`, `Rubus`, `Actinidia` are deciduous climbers/shrubs (included).

In [ ]:
# Genus-level inclusion list for deciduous trees/shrubs present in pep725_temp.
DECIDUOUS_GENERA = {
    'Acer', 'Actinidia', 'Aesculus', 'Alnus', 'Berberis', 'Betula', 'Carpinus',
    'Castanea', 'Catalpa', 'Cornus', 'Corylus', 'Crataegus', 'Cydonia',
    'Fagus', 'Forsythia', 'Fraxinus', 'Juglans', 'Larix', 'Magnolia',
    'Malus', 'Mespilus', 'Morus', 'Philadelphus', 'Platanus', 'Populus',
    'Prunus', 'Punica', 'Pyrus', 'Quercus', 'Ribes', 'Robinia', 'Rosa',
    'Rubus', 'Salix', 'Sambucus', 'Sorbus', 'Symphoricarpos', 'Syringa',
    'Tilia', 'Ulmus', 'Vaccinium', 'Vitis',
}
# Explicit exclusions — evergreen or non-tree even though genus partly fits
NON_DECIDUOUS_OVERRIDE = {
    'Ilex',  # not present anyway
}

def is_deciduous_tree(scientific_name: str) -> bool:
    g = _genus(scientific_name)
    return g in DECIDUOUS_GENERA and scientific_name not in NON_DECIDUOUS_OVERRIDE


deciduous_species = [sp for sp in species_with_bloom if is_deciduous_tree(sp)]
dropped = [sp for sp in species_with_bloom if not is_deciduous_tree(sp)]

print(f'Species classified as deciduous tree/shrub: {len(deciduous_species)}')
print(f'Dropped (herbaceous / evergreen / other):   {len(dropped)}')
print()
print('Kept:')
for sp in deciduous_species:
    print(f'  {sp}')
print('\nDropped:')
for sp in dropped:
    print(f'  {sp}')

In [ ]:
# Build a trimmed Observations and site-year frame for the deciduous set
species_names_dec = {
    key: name for key, name in species_names_bloom.items()
    if name in set(deciduous_species)
}
dec_sids = {k[1] for k in species_names_dec}

mask_dec = (
    df_raw.index.get_level_values(KEY_SPECIES_ID).isin(dec_sids)
    & (df_raw.index.get_level_values(KEY_OBS_TYPE) == TARGET_OBS)
)
df_y_dec = df_raw[mask_dec]
obs_dec = Observations(df_y_dec, df_y_loc, species_names=species_names_dec)

df_bloom_dec  = df_bloom[df_bloom['species_name'].isin(deciduous_species)].copy()
df_sy_dec = (
    df_bloom_dec
    .groupby([KEY_DATA_SOURCE, KEY_LOC_ID, KEY_YEAR, 'species_name'])['doy']
    .median()
    .reset_index()
)

print(f'Deciduous tree observations (BBCH_60): {len(df_bloom_dec):,}')
print(f'Site-year records:                      {len(df_sy_dec):,}')

## 8. Deciduous-tree phenology overview

In [ ]:
counts_dec = (df_bloom_dec.groupby('species_name')['doy']
              .agg(n='count', mean='mean', std='std', median='median')
              .sort_values('median'))
species_order_dec = counts_dec.index.tolist()

fig, ax = plt.subplots(figsize=(max(10, 0.35 * len(species_order_dec)), 5.5))
palette = _palette('tab20', len(species_order_dec))
doy_by_sp = [df_bloom_dec.loc[df_bloom_dec['species_name'] == sp, 'doy'].values
             for sp in species_order_dec]
parts = ax.violinplot(doy_by_sp, positions=range(len(species_order_dec)),
                      widths=0.85, showmedians=True, showextrema=False)
for pc, col in zip(parts['bodies'], palette):
    pc.set_facecolor(col); pc.set_alpha(0.7)
parts['cmedians'].set_color('black')
ax.set_xticks(range(len(species_order_dec)))
ax.set_xticklabels([f'{sp}\n(n={counts_dec.loc[sp,"n"]:.0f})' for sp in species_order_dec],
                   rotation=75, ha='right', fontsize=7.5)
ax.set_ylabel('Day of year (BBCH_60)')
ax.set_title(f'Flowering phenology — deciduous trees / shrubs ({len(species_order_dec)} species)',
             fontweight='bold')
for doy, month in _DOY_MONTH:
    ax.axhline(doy, color='lightgrey', lw=0.5, zorder=0)
plt.tight_layout()
plt.show()

print(counts_dec.round(1).to_string())

## 9. Fit phylogeny — deciduous subset

In [ ]:
result_dec = run_phylo_analysis(obs_dec, df_sy_dec, k_embed=16,
                                 n_perm=1999, label='deciduous trees')

n_dec = len(result_dec['label_list'])
print(f'\nSpecies in phylogeny: {n_dec}')
print(f'Pairs with shared site-years: {len(result_dec["pairs"])} / {n_dec * (n_dec - 1) // 2}')
for key, res in result_dec['mantel'].items():
    print(f"  Mantel {key:<9s}: stat={res['stat']:+.3f}  p={res['p_val']:.4f}")

In [ ]:
plot_distance_matrix(result_dec)

In [ ]:
linkage_dec, genus_colors_dec = plot_dendrogram(result_dec)

In [ ]:
plot_mds(result_dec, linkage_dec, genus_colors_dec, max_dims=4)

In [ ]:
plot_phylo_signal(result_dec)
plot_mantel(result_dec)

## 10. Side-by-side comparison

Does the phylogenetic signal in flowering phenology actually **sharpen** when we restrict to deciduous trees? Compare Mantel statistics and the pair-level scatter side by side.

In [ ]:
def _pair_arrays(result):
    pairs = result['pairs']
    x = np.array([p[4] for p in pairs])
    y = np.array([p[5] for p in pairs])
    n = np.array([p[6] for p in pairs])
    same_gen = np.array([_genus(p[0]) == _genus(p[1]) for p in pairs])
    return x, y, n, same_gen


fig, axes = plt.subplots(1, 2, figsize=(14, 5.5), sharey=True)

for ax, res in zip(axes, [result_all, result_dec]):
    x, y, n, same_gen = _pair_arrays(res)
    r = np.corrcoef(x, y)[0, 1] if len(x) > 1 else np.nan
    rho = np.corrcoef(_rankdata(x), _rankdata(y))[0, 1] if len(x) > 1 else np.nan
    ax.scatter(x[~same_gen], y[~same_gen], s=15 + (n[~same_gen]/max(n.max(),1))*120,
               color='#e74c3c', alpha=0.5, edgecolors='white', linewidths=0.4,
               label='diff genus')
    ax.scatter(x[same_gen],  y[same_gen],  s=15 + (n[same_gen]/max(n.max(),1))*120,
               color='#2ecc71', alpha=0.7, edgecolors='white', linewidths=0.4,
               label='same genus')
    if len(x) > 1:
        m = np.polyfit(x, y, 1)
        xr = np.linspace(x.min(), x.max(), 50)
        ax.plot(xr, np.polyval(m, xr), 'k--', lw=1.3)
    p_pearson  = res['mantel']['pearson']['p_val']
    p_spearman = res['mantel']['spearman']['p_val']
    ax.set_title(f'{res["label"]}  (n={len(res["label_list"])} species, {len(x)} pairs)\n'
                 f'r={r:+.2f} (Mantel p={p_pearson:.3f})   '
                 f'ρ={rho:+.2f} (Mantel p={p_spearman:.3f})',
                 fontsize=11, fontweight='bold')
    ax.set_xlabel('Phylogenetic distance (patristic)')
    ax.legend(fontsize=8.5)

axes[0].set_ylabel('Mean paired |ΔDOY|  (days)')
plt.tight_layout()
plt.show()

## 11. Summary

Key numbers from both analyses:

In [ ]:
def _summary_row(res, name):
    pairs = res['pairs']
    x = np.array([p[4] for p in pairs]) if pairs else np.array([])
    y = np.array([p[5] for p in pairs]) if pairs else np.array([])
    return {
        'Analysis':              name,
        'Species':               len(res['label_list']),
        'Pairs (shared loc×yr)': len(pairs),
        'Pearson r':             f"{np.corrcoef(x, y)[0,1]:+.3f}" if len(x) > 1 else '—',
        'Mantel p (Pearson)':    f"{res['mantel']['pearson']['p_val']:.4f}",
        'Spearman ρ':            f"{np.corrcoef(_rankdata(x), _rankdata(y))[0,1]:+.3f}" if len(x) > 1 else '—',
        'Mantel p (Spearman)':   f"{res['mantel']['spearman']['p_val']:.4f}",
    }

summary_df = pd.DataFrame([
    _summary_row(result_all, 'All bloom species'),
    _summary_row(result_dec, 'Deciduous trees'),
]).set_index('Analysis')
print(summary_df.to_string())